In [ ]:
import pandas as pd

# ========================
# EXACT FILE PATHS YOU GAVE
# ========================
mapping_file = r"D:/Tushar/main_with_subs_only.xlsx"
indent_file  = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"

# ========================
# LOAD FILES
# ========================
df_mapping = pd.read_excel(mapping_file)
df_indent  = pd.read_excel(indent_file, sheet_name="Sheet1")

# ========================
# RENAME COLUMNS (using your exact names)
# ========================
df_mapping = df_mapping.rename(columns={
    'Main_Label': 'Child_Part',
    'Sub_Label':  'Switch_Part',
    'Sub_Count':  'Qty_per_Switch',      # qty of child used in one switch
    'Main_Count': 'Ignore_Historical'
})

df_indent = df_indent.rename(columns={
    'Part number': 'Switch_Part'
})

# ========================
# EXACT MONTH COLUMNS FROM YOUR FILE (Sheet1)
# ========================
month_cols = ["Feb'26 QTY", "Mar'26", "Apr'26", "May'26", "Jun'26", "Jul'26"]

# Clean names for new calculated columns
clean_months = ["Feb26", "Mar26", "Apr26", "May26", "Jun26", "Jul26"]

# ========================
# MERGE
# ========================
df_merged = pd.merge(
    df_mapping[['Child_Part', 'Switch_Part', 'Qty_per_Switch']],
    df_indent[['Switch_Part'] + month_cols],
    on='Switch_Part',
    how='left'
)

print(f"Total rows after merge: {len(df_merged)}")
missing = df_merged[month_cols[0]].isna().sum()
print(f"Missing data in {month_cols[0]}: {missing} rows (should be 0 or very low)")

# ========================
# CALCULATE DAILY + 2-DAYS REQUIREMENT
# ========================
for month, clean in zip(month_cols, clean_months):
    daily_col   = f"Daily_{clean}"
    twodays_col = f"2Days_{clean}"
    
    df_merged[daily_col]   = (df_merged[month] / 30.0).round(2)
    df_merged[twodays_col] = (df_merged[daily_col] * df_merged['Qty_per_Switch'] * 2).round(2)

# ========================
# 1. TOTALS PER CHILD PART
# ========================
totals = df_merged.groupby('Child_Part', as_index=False).agg({
    f"Daily_{clean}": 'sum' for clean in clean_months
})

for clean in clean_months:
    totals[f"2Days_{clean}"] = (totals[f"Daily_{clean}"] * 2).round(2)

totals = totals[
    ['Child_Part'] +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
]

totals.to_excel("Child_Totals_2Days_Per_Month.xlsx", index=False)
print(f"✅ Totals file saved: Child_Totals_2Days_Per_Month.xlsx ({len(totals)} rows)")

# ========================
# 2. DETAILED BREAKDOWN
# ========================
detailed_cols = (
    ['Child_Part', 'Switch_Part', 'Qty_per_Switch'] +
    month_cols +
    [f"Daily_{clean}" for clean in clean_months] +
    [f"2Days_{clean}" for clean in clean_months]
)

detailed = df_merged[detailed_cols]
detailed.to_excel("Child_Detailed_Breakdown_2Days.xlsx", index=False)
print(f"✅ Detailed file saved: Child_Detailed_Breakdown_2Days.xlsx ({len(detailed)} rows)")

print("\n🎉 ALL DONE! Check the two new Excel files in your folder.")